## Data Ingestion

### Basic CSV Loader Outline (Reference )

In [ ]:
# from langchain_community.document_loaders import CSVLoader

# loader = CSVLoader(
#     file_path="../data/text_files/njit_orgs.csv",
#     encoding="utf-8"
# )

# docs = loader.load()
# print(docs[0].page_content[:100])
# print(docs[0].metadata)

Index: id
Height: source_type
Weight: title
None: description,category,tags,metadata_json,url,last_u
{'source': '../data/text_files/njit_orgs.csv', 'row': 0}


### Entire Directory Ingestion ( Reference )

In [ ]:
# import os
# from langchain_community.document_loaders import DirectoryLoader

# os.makedirs("../data/text_files", exist_ok=True)

# dir_loader = DirectoryLoader(
#     "../data/text_files",
#     glob="**/*.txt",
# )

# documents=dir_loader.load()
# documents

ImportError: cannot import name 'Directory' from 'langchain_community.document_loaders' (c:\Users\Moise\Documents\BoFA-JitHub\react-webapp\backend\rag\.venv\Lib\site-packages\langchain_community\document_loaders\__init__.py)

## Injestion ( Beginning of Pipeline )

### Read all files(PDF,CSV or TXT) inside Directory

In [1]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    CSVLoader,
    PyPDFLoader,
)
from pathlib import Path
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO) #Setup for root logging showing INFO, WARNING, ERROR
logger = logging.getLogger("ingestion")

DATA_DIR= Path("../data/text_files") #Replaced old os import for file path
DATA_DIR.mkdir(parents=True, exist_ok=True) #Replaced original, creates directry and any missing parent directories

# Map extensions to their loader class + kwargs
# Add/Remove entries as your actual sources evolve

LOADER_REGISTRY = { #dictionary maps file extension to tuple (loader_class, loader_kwargs)
    #loader_kwargs = dictionary of extra arguments passed to that loader class when instantiated
    ".txt": (TextLoader, {"autodetect_encoding": True}),
    ".csv": (CSVLoader, {"encoding": "utf-8", "csv_args": {"delimiter" : ","}}),
    ".pdf": (PyPDFLoader, {"encoding": "utf-8"}),
}

def load_documents(data_dir: Path):

    all_docs = []

    #Global mismatches silently
    #Intead of single hardcoded global variable name (i.e csv or pdf)
    #get explicitly ignored-with-a-log, not silently dropped

    for ext, (loader_cls, loader_kwargs) in LOADER_REGISTRY.items():
        loader = DirectoryLoader(
            str(data_dir),
            glob=f"**/*{ext}",
            loader_cls=loader_cls,
            loader_kwargs=loader_kwargs,
            recursive=True, #flag to walk subdirectories, rather than relying on glob pattern alone
            silent_errors=True, #If one file throws during loading(encoding error, corrupt PDF, etc.) DirectoryLoader logs a warning internally and skips
            show_progress=True, #Shows a progress bar (via tqdm) while loading
        )
        try:
            docs = loader.load()
        except Exception as e:
            logger.errror(f"Loader failed entirely for {ext}: {e}")
            continue

        if docs:
            logger.info(f"Loaded {len(docs)} {ext} document(s)")
            all_docs.extend(docs)
        else:
            logger.warning(f"No {ext} files found in {data_dir}")
        
        #Handle empty directory / no matches ( fail loudly )
        #Dont let chunking/embedding run on zero docs!!!!

    if not all_docs:
        raise ValueError(
            f"No documents loaded from {data_dir}."
            f"Checked extensions: {list(LOADER_REGISTRY.keys())}. "
            f"Confirm the directory isn't empty and files match a known type"
        )
    
    #Warn on large batches before chunking/embedding,
    #decide to batch/stream instead

    if len(all_docs) > 5000:
        logger.warning(
            f"loaded {len(all_docs)} documents. Consider streaming/batching"
        )
    return all_docs

def flag_unmatched_files(data_dir: Path):
    #Report files that dont match any known extension, instead of DirectoryLoader silently ignoring

    known_exts = set(LOADER_REGISTRY.keys())
    unmatched = [
        p for p in data_dir.rglob("*")
        if p.is_file() and p.suffix.lower() not in known_exts
    ]
    if unmatched:
        logger.warning(
            f"{len(unmatched)} file(s) with no matching loader: "
            f"{[str(p) for p in unmatched[:10]]}"
            f"{'...' if len(unmatched) > 10 else ''}"
        )

if __name__ == "__main__":
    flag_unmatched_files(DATA_DIR)
    documents = load_documents(DATA_DIR)
    logger.info(f"Total documents loaded: {len(documents)}")

C:\Users\Moise\AppData\Local\Temp\ipykernel_8576\3580779994.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
c:\Users\Moise\Documents\BoFA-JitHub\react-webapp\backend\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
0it [00:00, ?it/s]
100%|██████████| 1/1 [00:00<00:00, 146.73it/s]
INFO:ingestion:Loaded 282 .csv document(s)
0it [00:00, ?it/s]
INFO:ingestion:Total documents loaded: 282


## Text Splitting into Chunks ( Chunking )

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    ''' Split documents into smaller chunks for better RAG performance '''
    text_splitter = RecursiveCharacterTextSplitter( 
        chunk_size = chunk_size, #max size of chunk, determined by length_function
        chunk_overlap = chunk_overlap, #target overlap between chunks
        length_function = len, #function determining chunk size
        separators=["\n\n", "\n", " ", ""] #Whether the seperator list should be interpreted as regex
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    return split_docs


## Embedding and vectorStore DB

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from dotenv import load_dotenv

load_dotenv()

embedding=OpenAIEmbeddings()
vector_store=FAISS.from_documents(split_documents(load_documents(DATA_DIR)), embedding)


0it [00:00, ?it/s]
100%|██████████| 1/1 [00:00<00:00, 139.90it/s]
INFO:ingestion:Loaded 282 .csv document(s)
0it [00:00, ?it/s]


Split 282 documents into 548 chunks


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.


## Retriver Pipeline

In [5]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()


llm = ChatOpenAI(model="gpt-3.5-turbo") #gpt-4

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever()
)

query = "What is Society of Hispanic Professional Engineers"
response = qa_chain.invoke(query)
print(response)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'query': 'What is Society of Hispanic Professional Engineers', 'result': 'The Society of Hispanic Professional Engineers (SHPE) is an organization that holds professional events throughout the semesters to help students grow professionally and prepare them for the corporate world. SHPE also focuses on maintaining a family bond with members to provide a support system through their journey in college. They empower the NJIT community to realize their fullest potential and impact the world through STEM awareness, access, support, and development. You can find more information about SHPE on their organization profile: [Society of Hispanic Professional Engineers](https://njit.campuslabs.com/engage/organization/shpe).'}
